In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 04 — Data Quality Gate
# MAGIC Validates the whole run, including a self-diagnosing check for a real streaming
# MAGIC gotcha: duplicate window emissions from `outputMode("update")`.

# COMMAND ----------

import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))

# COMMAND ----------

from iot_pipeline.config import BRONZE_TABLE, SILVER_TABLE, GOLD_TABLE

checks = []

# COMMAND ----------

bronze_count = spark.table(BRONZE_TABLE).count()
silver_count = spark.table(SILVER_TABLE).count()
gold_count = spark.table(GOLD_TABLE).count()

checks.append(("bronze_not_empty", bronze_count > 0))
checks.append(("silver_not_empty", silver_count > 0))

# COMMAND ----------

# Gold alerts should only ever come from sensor_3 in this seed dataset
bad_alerts = spark.sql(f"""
    SELECT count(*) AS n FROM {GOLD_TABLE} WHERE device_id != 'sensor_3'
""").collect()[0]["n"]
checks.append(("gold_alerts_only_sensor_3", bad_alerts == 0))

# COMMAND ----------

# Self-diagnosing check: no duplicate (window, device_id) pairs in Silver
dupe_windows = spark.sql(f"""
    SELECT window, device_id, count(*) AS n FROM {SILVER_TABLE}
    GROUP BY window, device_id HAVING count(*) > 1
""").count()
checks.append(("no_duplicate_silver_windows", dupe_windows == 0))

# COMMAND ----------

failed = [name for name, passed in checks if not passed]
for name, passed in checks:
    print(f"{'✅' if passed else '❌'} {name}")

if "no_duplicate_silver_windows" in failed:
    print("\n⚠️ Duplicate windows found — Structured Streaming processed data across")
    print("   multiple micro-batches and emitted more than one update for the same")
    print("   window. See Part 18 (Level Up) in the guide for the foreachBatch + MERGE fix.")

assert not failed, f"Data quality checks failed: {failed}"
print("\n✅ All data quality checks passed.")